# 🦙 LangChain 100% Local con Ollama

![Python](https://img.shields.io/badge/Python-3.10%2B-blue)
![LangChain](https://img.shields.io/badge/LangChain-0.3.x-1C3C3C)
![Ollama](https://img.shields.io/badge/Ollama-local-black)
![License](https://img.shields.io/badge/license-MIT-green)

## 📖 Sobre este notebook

Este notebook es una **adaptación 100% local** del lab de IBM/Skills Network
*"Build Smarter AI Apps: Empower LLMs with LangChain"*. El lab original corre
en JupyterLab de Skills Network y depende de la API de **watsonx.ai** de IBM
(sin costo pero requiere el entorno web de IBM). Cuando ese entorno dejó de
arrancar el kernel, se rehízo el mismo recorrido conceptual usando:

- **[Ollama](https://ollama.com/)** como servidor de modelos local (sin API keys, sin costo, sin depender de un proveedor externo).
- **`llama3.2`** como modelo de chat.
- **`nomic-embed-text`** como modelo de embeddings.
- **ChromaDB** como vector store, igual que el lab original.

El objetivo no es solo "que el código corra", sino entender **qué hace cada
pieza y por qué**, incluyendo los tropiezos reales que aparecen al usar un
modelo pequeño corriendo en hardware local (limitaciones de tool-calling,
no determinismo, sensibilidad al tamaño de chunk, etc.) — cosas que casi
nunca se ven con modelos grandes de pago y que son justo las que importan
cuando se construye algo para producción con recursos limitados.

## ✅ Requisitos previos

1. **[Ollama](https://ollama.com/download)** instalado y corriendo (`ollama serve`, normalmente arranca solo).
2. Descargar los dos modelos usados en este notebook:
   ```bash
   ollama pull llama3.2
   ollama pull nomic-embed-text
   ```
3. Python 3.10+ y un entorno virtual (recomendado):
   ```bash
   python -m venv .venv
   source .venv/bin/activate  # En Windows: .venv\Scripts\activate
   ```

## 📦 Instalación de dependencias

Ejecuta la primera celda de código de este notebook, o instala manualmente:

```bash
pip install langchain langchain-core langchain-community langchain-ollama \
            langchain-text-splitters langchain-chroma chromadb pydantic pypdf
```

## 🗂️ Contenido

1. Modelo de chat local (`ChatOllama`)
2. Roles de mensajes (`System` / `Human` / `AI`)
3. Prompt templates dinámicos
4. Salida estructurada con Pydantic (`with_structured_output`)
5. El objeto `Document`
6. Document loaders (PDF)
7. Text splitters
8. Embeddings locales (`OllamaEmbeddings`)
9. Vector store (`Chroma`) y búsqueda por similitud
10. Retrievers y RAG (Retrieval-Augmented Generation) con LCEL
11. Memoria conversacional (`RunnableWithMessageHistory`)
12. Chains secuenciales con LCEL
13. Tools y agentes (tool calling nativo)
14. Conclusiones y recomendaciones generales

## ⚠️ Nota honesta sobre modelos pequeños locales

A lo largo del notebook vas a encontrar comentarios de tipo "esto puede
fallar con llama3.2" o "esto no es 100% determinista". Es intencional: parte
del valor de correr esto en local con un modelo de 3B parámetros es
**ver en vivo** las limitaciones que no se notan con modelos grandes de
pago (GPT-4, Claude, Llama 4 Maverick, etc.), y aprender a diagnosticarlas
en vez de asumir que "el código está mal".

---


## 1. Instalación de dependencias

Se agrega `langchain-chroma` (el paquete dedicado y mantenido para Chroma)
en vez de importar `Chroma` desde `langchain_community`, que está en
proceso de sunset y lanza `DeprecationWarning` (lo viste en la ejecución
original). El comportamiento es idéntico, solo cambia el import.

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-ollama \
                langchain-text-splitters langchain-chroma chromadb pydantic pypdf

## 2. Modelo de chat local

### Celda 1: Inicialización del modelo (`ChatOllama`)

Instanciamos el modelo de chat que usaremos en **todo** el notebook. En el
lab original esto requería `ModelInference` + credenciales de watsonx y un
wrapper `WatsonxLLM()`; aquí `ChatOllama` ya es un wrapper nativo de
LangChain que habla directo con tu servidor Ollama local (`localhost:11434`
por defecto).

- `model="llama3.2"`: debe estar descargado (`ollama pull llama3.2`).
- `temperature=0.2`: qué tan determinista/creativo es el modelo. Cerca de 0
  = respuestas más consistentes; cerca de 1 = más variadas.

> **Recomendación:** si vas a reproducir este notebook para un caso de uso
> real, sube la temperatura solo en las celdas de generación creativa y
> bájala (o déjala en 0) en las de clasificación / extracción estructurada,
> donde la consistencia importa más que la creatividad.

In [ ]:
# ==============================================================================
# CELDA 1: Inicialización del modelo local Llama 3.2
# ==============================================================================
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

# Instanciamos el modelo local
llm = ChatOllama(
    model="llama3.2",
    temperature=0.2,
)

print("Modelo Llama 3.2 cargado correctamente y listo para recibir peticiones.")

### Celda 2: Roles de mensajes (`System` / `Human` / `AI`)

Un chat model no recibe un string plano: recibe una **lista de mensajes con
roles**. `SystemMessage` define el comportamiento del asistente,
`HumanMessage` es la entrada del usuario, `AIMessage` representa una
respuesta del modelo (normalmente no la escribes tú, salvo para simular
historial).

**Punto clave a notar en la salida:** `respuesta` no es un string, es un
objeto `AIMessage`. Por eso se imprime `respuesta.content` y no `respuesta`
directamente — el objeto trae además metadata útil para debugging
(tokens usados, duración, etc.) que verás más adelante en la sección de
tools.

In [ ]:
# ==============================================================================
# CELDA 2: Gestión de Roles y Mensajes (System, Human, AI)
# ==============================================================================

# 1. Definimos la lista de mensajes emulando un diálogo
mensajes = [
    # SystemMessage: Define el rol, comportamiento y restricciones del modelo.
    SystemMessage(
        content="Eres un asistente especializado en Inteligencia Artificial y MLOps. "
        "Responde de forma clara, concisa y técnica."
    ),
    # HumanMessage: Representa la entrada o pregunta del usuario.
    HumanMessage(
        content="Explica brevemente qué es la cuantización de un modelo de LLM."
    ),
]

# 2. Invocamos al modelo pasando la lista completa de mensajes
respuesta = llm.invoke(mensajes)  # -> AIMessage

# 3. Analizamos la salida
print("--- TIPO DE OBJETO DEVUELTO ---")
print(type(respuesta))  # Devuelve un objeto AIMessage

print("\n--- RESPUESTA DEL MODELO ---")
print(respuesta.content)

## 3. Prompt templates dinámicos

### Celda 3: `ChatPromptTemplate`

Resuelve un problema concreto: no quieres escribir el prompt a mano cada vez
que cambian las variables. La plantilla usa placeholders (`{area}`,
`{concepto}`, `{tecnologia}`) que se sustituyen con `.invoke(dict)`.

El patrón **plantilla → formatear → invocar modelo** es la base conceptual
de las *chains* (`prompt | llm | parser`) que se usan más adelante en el
notebook (Sección 10 y 12): son literalmente los mismos pasos encadenados
con el operador `|`.

In [ ]:
# ==============================================================================
# CELDA 3: Prompts Dinámicos con ChatPromptTemplate
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate

# 1. Definimos la plantilla con variables entre llaves { }
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {area}. Responde de forma técnica y breve."),
    ("human", "Explica el concepto de {concepto} en el contexto de {tecnologia}."),
])

# 2. Formateamos el prompt pasando los valores de las variables
prompt_formateado = prompt_template.invoke({
    "area": "MLOps",
    "concepto": "Feature Store",
    "tecnologia": "arquitecturas de datos",
})

print("--- MENSAJES GENERADOS A PARTIR DE LA PLANTILLA ---")
print(prompt_formateado.to_messages())

print("\n--- RESPUESTA DEL MODELO ---")
# 3. Le pasamos el prompt formateado directamente al LLM
respuesta_dinamica = llm.invoke(prompt_formateado)
print(respuesta_dinamica.content)

## 4. Salida estructurada con Pydantic

### Celda 4: `with_structured_output`

Este es un salto de nivel respecto al `JsonOutputParser` que usa el lab
original de IBM (que mete instrucciones de formato como texto dentro del
prompt y "espera" que el modelo cumpla). `with_structured_output` usa el
mecanismo nativo de *tool calling* del modelo para **forzar** la respuesta
a encajar en el schema de Pydantic — el resultado ya no es un string ni un
dict, es una instancia real de `TecnicoMLSchema`, con tipos validados.

**Para qué sirve en la práctica:** cualquier caso donde la respuesta se
vaya a usar en código (guardar en una base de datos, llenar un frontend,
alimentar el siguiente paso de un pipeline) necesita campos separados y
predecibles, no un párrafo de texto libre del que hay que "adivinar" dónde
corta cada dato.

> **Requisito importante:** esto depende de que el modelo soporte *tool
> calling*. `llama3.2` lo soporta. Si migras a otro modelo local que no lo
> soporte, esta celda puede fallar o degradar silenciosamente — la primera
> sospecha en ese caso debería ser "¿este modelo soporta tool calling en
> Ollama?", no "está mal mi código".

In [ ]:
# ==============================================================================
# CELDA 4 (Enfoque estructurado nativo): Uso de with_structured_output
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# 1. Definimos el esquema de salida con Pydantic
class TecnicoMLSchema(BaseModel):
    tecnologia: str = Field(description="Nombre de la tecnología o concepto")
    categoria: str = Field(
        description="Categoría (ej: MLOps, Data Engineering, Infrastructure)"
    )
    descripcion_corta: str = Field(description="Explicación en una sola oración")
    ventaja_principal: str = Field(
        description="Beneficio clave de su implementación"
    )


# 2. Creamos una versión del modelo forzada a devolver el esquema
llm_estructurado = llm.with_structured_output(TecnicoMLSchema)

# 3. Definimos un prompt simple y directo
prompt_json = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un experto en arquitectura de datos y MLOps. Completa los campos solicitados.",
    ),
    ("human", "Analiza la tecnología: {tecnologia}"),
])

# 4. Construimos la cadena (Prompt | LLM Estructurado)
chain_json = prompt_json | llm_estructurado

# 5. Ejecutamos
resultado = chain_json.invoke({"tecnologia": "Feature Store"})

print("--- TIPO DE DATO DEVUELTO ---")
print(type(resultado))  # Devuelve un objeto de la clase TecnicoMLSchema

print("\n--- OBJETO RESULTANTE ---")
print(resultado)

print("\n--- ACCESO A CAMPOS ---")
print(f"Tecnología: {resultado.tecnologia}")
print(f"Categoría: {resultado.categoria}")
print(f"Ventaja principal: {resultado.ventaja_principal}")

> 💡 **Recomendación:** prueba esta celda con 2-3 valores distintos de
> `tecnologia` ("Kubernetes", "Vector Database", "CI/CD Pipeline") para
> confirmar que el schema se respeta de forma consistente. Es una buena
> forma de medir qué tan confiable es `with_structured_output` con un
> modelo pequeño antes de depender de él en un pipeline real.

## 5. Documents: el bloque de datos de LangChain

### Celda 5: El objeto `Document`

Antes de cargar nada, vale la pena ver la forma más simple de un
`Document`: un contenedor con **`page_content`** (el texto) y
**`metadata`** (info sobre de dónde salió ese texto). Todo lo que sigue
(loaders, splitters, vector stores) básicamente produce montones de estos
objetos automáticamente.

In [ ]:
# ==============================================================================
# CELDA 5: El objeto Document (estructura básica)
# ==============================================================================
from langchain_core.documents import Document

doc_ejemplo = Document(
    page_content="""Python es un lenguaje de programación interpretado de alto nivel.
    Su filosofía de diseño enfatiza la legibilidad del código.""",
    metadata={
        "fuente": "ejemplo manual",
        "autor": "yo",
        "id": 1
    }
)

print("--- CONTENIDO ---")
print(doc_ejemplo.page_content)

print("\n--- METADATA ---")
print(doc_ejemplo.metadata)

### Celda 6: Document loader (PDF desde URL)

Cargamos el mismo paper que usa el lab original de IBM
(*"Revolutionizing Mental Health Care through LangChain"*), para tener un
punto de comparación directo. `PyPDFLoader.load()` devuelve una **lista de
`Document`**, uno por página, con metadata generada automáticamente
(`source`, `page`, `total_pages`, etc.) — no la escribes tú, el loader la
genera.

Este paso no usa ningún LLM todavía: es pura ingeniería de datos (bajar el
PDF y extraer el texto).

In [ ]:
# ==============================================================================
# CELDA 6: Document loader - PDF desde una URL
# ==============================================================================
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

print(f"Número de páginas cargadas: {len(document)}")
print("\n--- METADATA DE LA PÁGINA 0 ---")
print(document[0].metadata)
print("\n--- PRIMEROS 500 CARACTERES DE LA PÁGINA 1 ---")
print(document[1].page_content[:500])

### Celda 7: Text splitter (`RecursiveCharacterTextSplitter`)

**Corrección respecto a la versión original de esta celda:** los
comentarios decían "~200 caracteres" y "20 de overlap" pero los parámetros
reales ya eran `chunk_size=800` / `chunk_overlap=100` — quedaron
desincronizados en una iteración anterior. Se corrigen los comentarios
abajo para que coincidan con los valores reales.

También se cambia `CharacterTextSplitter` por `RecursiveCharacterTextSplitter`,
que fue el ajuste que se validó como mejora real durante las pruebas de RAG
más adelante en este notebook (Sección 10): corta primero por párrafos,
luego oraciones, luego palabras, en ese orden — en vez de cortar ciegamente
cada N caracteres — lo que reduce las oraciones partidas a la mitad y
mejora notablemente la calidad de las respuestas de RAG con `chunk_size`
pequeños como 200.

**Trade-off de `chunk_size` a tener en cuenta:**
- Muy chico → se pierde contexto (frases cortadas a la mitad).
- Muy grande → el embedding se vuelve "difuso", mezclando demasiadas ideas
  en un solo vector.

`chunk_overlap` repite los últimos N caracteres de un chunk al inicio del
siguiente, para que una idea que cae justo en el borde de un corte no se
pierda por completo.

In [ ]:
# ==============================================================================
# CELDA 7: Text splitter (recursivo, por párrafos/oraciones)
# ==============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # cada chunk tendrá ~800 caracteres
    chunk_overlap=100,   # 100 caracteres de traslape entre chunks consecutivos
)

chunks = text_splitter.split_documents(document)

print(f"Número total de chunks: {len(chunks)}")
print("\n--- CONTENIDO DEL CHUNK #5 ---")
print(chunks[5].page_content)
print("\n--- METADATA DEL CHUNK #5 ---")
print(chunks[5].metadata)

> 💡 **Recomendación:** si vas a reutilizar este notebook con otro PDF
> más largo o técnico, prueba 2-3 combinaciones de `chunk_size` /
> `chunk_overlap` y compara la calidad de las respuestas de RAG antes de
> quedarte con un valor fijo — no hay un tamaño "correcto" universal, depende
> del documento y del tipo de pregunta que vas a hacer.

## 6. Embeddings locales

### Celda 8: `OllamaEmbeddings`

Aquí está el primer cambio real de proveedor respecto al lab de IBM (que
usa `WatsonxEmbeddings` con `ibm/granite-embedding-278m-multilingual`).
Un embedding convierte texto en un vector numérico de forma que textos con
significado similar terminan con vectores cercanos — eso es lo que permite
la búsqueda semántica más adelante.

Requiere el modelo de embeddings descargado en Ollama:
```bash
ollama pull nomic-embed-text
```

El resto del pipeline (vector store, retriever, RAG) funciona exactamente
igual una vez que tienes estos vectores, sin importar si vinieron de
watsonx o de Ollama — es la pieza más intercambiable de todo el lab.

In [ ]:
# ==============================================================================
# CELDA 8: Embeddings locales con Ollama
# ==============================================================================
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

# Extraemos solo el texto de cada chunk (igual que en el lab original)
texts = [chunk.page_content for chunk in chunks]

# Vectorizamos todos los chunks
embedding_result = embedding_model.embed_documents(texts)

print(f"Número de vectores generados: {len(embedding_result)}")
print(f"Dimensión de cada vector: {len(embedding_result[0])}")
print("\n--- PRIMEROS 5 NÚMEROS DEL PRIMER VECTOR ---")
print(embedding_result[0][:5])

## 7. Vector store y búsqueda semántica

### Celda 9: Chroma

**Corrección:** se importa `Chroma` desde `langchain_chroma` en vez de
`langchain_community.vectorstores`. El paquete de `langchain_community`
está en proceso de sunset para varias integraciones (Chroma incluida) y
emite `DeprecationWarning` — el comportamiento es idéntico, solo cambia de
dónde se importa.

In [ ]:
# ==============================================================================
# CELDA 9: Vector store con Chroma
# ==============================================================================
from langchain_chroma import Chroma

docsearch = Chroma.from_documents(chunks, embedding_model)

print("Vector store creado correctamente.")

### Celda 10: `similarity_search`

`docsearch.similarity_search(query)` vectoriza tu query con el mismo
`embedding_model` y devuelve los chunks cuyo vector está matemáticamente
más cerca — es decir, los más relacionados en **significado**, no
necesariamente los que contienen literalmente esas palabras.

In [ ]:
# ==============================================================================
# CELDA 10: Similarity search
# ==============================================================================
query = "Langchain"
docs = docsearch.similarity_search(query)

print(f"Número de resultados: {len(docs)}")
print("\n--- CHUNK MÁS SIMILAR A LA QUERY ---")
print(docs[0].page_content)

### Celda 11: Búsqueda por concepto (no por palabra literal)

Esta celda es la prueba real de que la búsqueda es **semántica**: la query
no contiene ninguna palabra literal del texto en inglés, y aun así debería
recuperar el chunk correcto sobre salud mental — la demostración de que
Chroma está comparando *significado*, no haciendo un `Ctrl+F`.

In [ ]:
# ==============================================================================
# CELDA 11: Similarity search por concepto (sin coincidencia literal de palabras)
# ==============================================================================
docs = docsearch.similarity_search("¿cómo se usa esto en salud mental?")
print(docs[0].page_content)

### Celda 12: El `retriever`

Un **retriever** es una interfaz más genérica que `similarity_search()`
directo: estandariza "recibe un string, devuelve una lista de `Document`",
lo que permite intercambiar la estrategia de búsqueda (similarity search,
MMR, otro vector store, etc.) sin cambiar el resto del pipeline. Aquí el
resultado es idéntico al de la búsqueda directa — el valor está en la
interfaz común, no en el resultado de esta celda en particular.

In [ ]:
# ==============================================================================
# CELDA 12: Retriever
# ==============================================================================
retriever = docsearch.as_retriever()

docs = retriever.invoke("mental health")
print(docs[0].page_content)

## 8. RAG (Retrieval-Augmented Generation) con LCEL

### Celda 13: Chain de RAG

El lab original usa `RetrievalQA.from_chain_type(...)`, que vive en
`langchain.chains` — un módulo que, según la versión instalada de
`langchain`, puede no estar disponible o requerir actualizar el paquete
(`ModuleNotFoundError: No module named 'langchain.chains'` es un error real
que se puede reproducir con instalaciones parciales). Además, el propio lab
de IBM advierte que las *chains* tradicionales están en camino a quedar
obsoletas en favor de **LCEL**.

Por ambas razones, se construye el RAG directamente con LCEL, sin depender
de `RetrievalQA`:

- `retriever | formatear_docs`: busca los chunks relevantes y los une en un
  solo string de contexto.
- `RunnablePassthrough()`: deja pasar la pregunta original sin tocarla.
- Ambas ramas corren en paralelo sobre el mismo input y arman el
  diccionario `{"context": ..., "question": ...}` que necesita `rag_prompt`.

> ⚠️ **Limitación importante de RAG por similarity search, confirmada más
> abajo con evidencia real:** funciona muy bien para preguntas puntuales
> ("¿qué aplicación de LangChain se menciona para salud mental?"), porque
> ahí sí existe un chunk que literalmente contiene esa respuesta. Funciona
> **mal** para preguntas de resumen global ("¿de qué trata este paper?"),
> porque ninguna pregunta tipo "resume esto" se parece semánticamente a
> ningún fragmento específico del texto — el retriever termina trayendo
> chunks dispersos y poco representativos.

In [ ]:
# ==============================================================================
# CELDA 13: RAG con LCEL
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Prompt que combina el contexto recuperado + la pregunta del usuario
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde la pregunta basándote únicamente en el siguiente contexto:\n\n{context}"),
    ("human", "{question}")
])

# 2. Función para convertir los chunks recuperados en un solo string de contexto
def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Armamos la chain RAG completa
rag_chain = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 4. La probamos con una pregunta de RESUMEN GLOBAL (caso débil de RAG, ver nota arriba)
resultado = rag_chain.invoke("¿De qué trata este paper?")
print(resultado)

### Celda 14: Diagnóstico — ¿qué recuperó realmente el retriever?

Antes de "arreglar" nada, conviene inspeccionar qué chunks trajo el
retriever para la pregunta de resumen. En las pruebas de este notebook,
para esta pregunta el retriever trajo fragmentos dispersos y poco
relevantes (referencias bibliográficas, pies de figura rotos, contenido
sobre Streamlit) — ninguno resume bien el paper, lo cual explica por qué la
respuesta de la Celda 13 fue vaga o parcialmente incorrecta.

In [ ]:
# ==============================================================================
# CELDA 14: Diagnóstico - inspeccionar los chunks recuperados
# ==============================================================================
docs_debug = retriever.invoke("¿De qué trata este paper?")
for i, d in enumerate(docs_debug):
    print(f"--- CHUNK {i} ---")
    print(d.page_content[:200])
    print()

### Celda 15: RAG con una pregunta puntual (caso fuerte de RAG)

Ahora probamos una pregunta específica, donde sí existe un chunk que
contiene la respuesta literal. Este es el escenario donde RAG por
similarity search rinde bien.

In [ ]:
# ==============================================================================
# CELDA 15: RAG con pregunta puntual (caso fuerte de retrieval)
# ==============================================================================
resultado = rag_chain.invoke("¿Qué aplicación de LangChain se menciona relacionada con salud mental?")
print(resultado)

### Celda 16: Cuando RAG NO es la herramienta correcta

Para un **resumen global** de un documento corto (este PDF son solo 6
páginas, cabe entero en el contexto de `llama3.2`), la solución correcta no
es "buscar el chunk más parecido a la pregunta" — es simplemente mandar
**todo el documento** de una sola vez. Esto **no es RAG**, es aprovechar
que el documento es corto.

> 📌 **Conclusión de esta sección:** RAG con retrieval por similarity
> search brilla cuando el documento es demasiado grande para caber en el
> contexto del modelo y la pregunta es puntual. Si el documento cabe entero
> y la pregunta es de resumen/síntesis global, pasar el contexto completo
> suele dar mejores resultados — y es más simple. En producción, un sistema
> maduro de RAG normalmente combina ambas estrategias según el tipo de
> pregunta y el tamaño del corpus, en vez de usar una sola receta fija.

In [ ]:
# ==============================================================================
# CELDA 16: Resumen usando el documento completo (SIN retrieval)
# ==============================================================================
texto_completo = "\n\n".join(doc.page_content for doc in document)

respuesta_resumen = llm.invoke(f"Resume de qué trata el siguiente paper:\n\n{texto_completo}")
print(respuesta_resumen.content)

## 9. Memoria conversacional

### Celda 17: `ChatMessageHistory` (concepto base)

Un wrapper ligero para acumular `HumanMessage` / `AIMessage` de una
conversación y recuperarlos después. Nada nuevo conceptualmente respecto a
la Celda 2 — la diferencia viene en cómo se conecta esto al LLM de forma
automática (siguiente celda).

In [ ]:
# ==============================================================================
# CELDA 17: Historial de mensajes básico
# ==============================================================================
from langchain_community.chat_message_histories import ChatMessageHistory

history = ChatMessageHistory()
history.add_ai_message("¡Hola! ¿En qué puedo ayudarte?")
history.add_user_message("¿Cuál es la capital de Francia?")

print(history.messages)

### Celda 18: Chatbot con memoria — enfoque moderno

El lab original usa `ConversationChain` + `ConversationBufferMemory`
(`langchain.chains` / `langchain.memory`), que además de depender del
mismo módulo problemático de la Celda 13, está en camino a quedar
deprecado. Se usa en su lugar `RunnableWithMessageHistory`, el patrón
recomendado actualmente:

1. `obtener_historial(session_id)` busca o crea el historial de esa sesión.
2. `RunnableWithMessageHistory` inyecta ese historial en el
   `MessagesPlaceholder("historial")` del prompt antes de invocar al
   modelo.
3. Después de la respuesta, guarda automáticamente tanto tu mensaje como la
   respuesta del modelo en ese mismo historial, para la próxima llamada.

El `session_id` permite manejar varias conversaciones/usuarios en paralelo
con el mismo objeto — útil si esto se convierte en una app con múltiples
usuarios.

In [ ]:
# ==============================================================================
# CELDA 18: Chatbot con memoria (enfoque moderno)
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. Prompt con un "hueco" para el historial
prompt_memoria = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil y conversacional."),
    MessagesPlaceholder("historial"),
    ("human", "{input}"),
])

# 2. Chain base (sin memoria todavía)
chain_base = prompt_memoria | llm

# 3. Almacén de historiales por sesión
almacen_sesiones = {}

def obtener_historial(session_id: str):
    if session_id not in almacen_sesiones:
        almacen_sesiones[session_id] = ChatMessageHistory()
    return almacen_sesiones[session_id]

# 4. Envolvemos la chain para que gestione el historial automáticamente
chain_con_memoria = RunnableWithMessageHistory(
    chain_base,
    obtener_historial,
    input_messages_key="input",
    history_messages_key="historial",
)

# 5. La probamos - nota el "config" con session_id
config = {"configurable": {"session_id": "usuario_1"}}

r1 = chain_con_memoria.invoke({"input": "Hola, soy un gatito pequeño. ¿Quién eres tú?"}, config=config)
print(r1.content)

r2 = chain_con_memoria.invoke({"input": "¿Quién soy yo?"}, config=config)
print(r2.content)

> 💡 **Verificación esperada:** en `r2`, el modelo debería recordar que
> dijiste que eras un gatito — eso confirma que el historial se está
> inyectando correctamente en cada llamada.

## 10. Chains secuenciales con LCEL

### Celda 19: ubicación → platillo → receta → tiempo de cocción

El lab original usa `SequentialChain` (mismo problema de dependencia y
deprecación que `RetrievalQA`/`ConversationChain`). Aquí se construye
directamente con `RunnablePassthrough.assign(...)`, que **agrega** una
nueva clave al diccionario que va viajando por la chain en cada paso, sin
borrar las anteriores — así cada paso puede ver los resultados de los
pasos previos.

> ⚠️ **Bug real encontrado y corregido durante el desarrollo de este
> notebook:** una primera versión de esta chain, con prompts más "sueltos"
> (sin instrucciones explícitas de formato), producía resultados
> encadenados incorrectos. La causa: la salida de `llm.invoke()` no
> siempre es texto "limpio" — el modelo a veces agrega relleno
> conversacional (p. ej. una pregunta de seguimiento tipo *"¿te gustaría
> saber más?"*) al final de su respuesta. Al meter esa salida cruda como
> input del siguiente paso, el modelo del siguiente paso interpretaba esa
> pregunta de seguimiento como *lo que se le está preguntando a él*, y
> terminaba respondiéndola en vez de generar el siguiente contenido
> esperado (por ejemplo, respondía "¡Claro que sí!" en vez de generar una
> receta).
>
> **La corrección:** instrucciones explícitas de formato en cada prompt
> ("Responde ÚNICAMENTE con...", "No hagas preguntas ni agregues
> comentarios extra"). Esta es la versión ya corregida y validada — con
> ella, cada paso genera contenido limpio y consistente con el paso
> anterior.
>
> **Lección para producción:** encadenar la salida cruda de un LLM como
> entrada del siguiente paso es riesgoso, especialmente con modelos
> pequeños. En pipelines reales, a veces vale la pena agregar un paso
> intermedio de "limpieza" (otra llamada al LLM, o incluso post-procesado
> con regex) entre etapas de una chain larga.

In [ ]:
# ==============================================================================
# CELDA 19: Sequential chain con LCEL (salidas "limpias")
# ==============================================================================
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Nota el "Responde ÚNICAMENTE con..." en cada plantilla: esto evita que el
# modelo agregue relleno conversacional que contamine el siguiente paso.
plantilla_ubicacion = PromptTemplate.from_template(
    "Responde ÚNICAMENTE con el nombre de un platillo clásico de: {ubicacion}. "
    "No agregues explicaciones, historia ni preguntas de seguimiento. Solo el nombre del platillo."
)
chain_ubicacion = plantilla_ubicacion | llm | StrOutputParser()

plantilla_receta = PromptTemplate.from_template(
    "Dame una receta corta y sencilla para hacer en casa: {platillo}. "
    "Responde SOLO con la receta (ingredientes y pasos). No hagas preguntas ni agregues comentarios extra."
)
chain_receta = plantilla_receta | llm | StrOutputParser()

plantilla_tiempo = PromptTemplate.from_template(
    "Dada la siguiente receta, estima el tiempo total de cocción. "
    "Responde SOLO con la estimación del tiempo, sin comentarios adicionales.\n\n{receta}"
)
chain_tiempo = plantilla_tiempo | llm | StrOutputParser()

chain_completa = (
    {"ubicacion": RunnablePassthrough()}
    | RunnablePassthrough.assign(platillo=chain_ubicacion)
    | RunnablePassthrough.assign(receta=lambda x: chain_receta.invoke({"platillo": x["platillo"]}))
    | RunnablePassthrough.assign(tiempo=lambda x: chain_tiempo.invoke({"receta": x["receta"]}))
)

resultado = chain_completa.invoke("México")

print("--- PLATILLO ---")
print(resultado["platillo"])
print("\n--- RECETA ---")
print(resultado["receta"])
print("\n--- TIEMPO ---")
print(resultado["tiempo"])

## 11. Tools y agentes

### Celda 20: Definir herramientas (`@tool`)

Hasta ahora el LLM solo generaba texto. Con **tools**, el modelo puede
decidir **ejecutar acciones** (calculadora, consulta de clima, etc.) y usar
el resultado para responder. El decorador `@tool` toma el **docstring** de
la función y lo convierte en la descripción que el modelo lee para decidir
cuándo usar esa herramienta — el docstring no es decorativo, es la
instrucción real que guía la decisión del modelo.

In [ ]:
# ==============================================================================
# CELDA 20: Definir herramientas (tools)
# ==============================================================================
from langchain_core.tools import tool

@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática simple, como '345 * 789' o '256 ** 0.5'."""
    try:
        return str(eval(expresion))
    except Exception as e:
        return f"Error al calcular: {e}"

@tool
def clima(ubicacion: str) -> str:
    """Devuelve el clima actual de una ubicación dada."""
    # En una app real, aquí llamarías a una API de clima de verdad.
    return f"El clima en {ubicacion} está soleado, 24°C."

herramientas = [calculadora, clima]

### Celda 21: `bind_tools` — el modelo decide, no ejecuta

`llm.bind_tools(herramientas)` "ata" las herramientas al modelo. Al
invocarlo, la respuesta **no es la respuesta final**: es una decisión de
qué herramienta llamar y con qué argumentos (`respuesta.tool_calls`). El
modelo no ejecuta la función — solo decide que debería usarse. Ejecutarla
de verdad es trabajo del código (siguiente celda) o de un agente que
automatice ese paso.

In [ ]:
# ==============================================================================
# CELDA 21: Bind tools al LLM
# ==============================================================================
llm_con_tools = llm.bind_tools(herramientas)

respuesta = llm_con_tools.invoke("¿Cuánto es 345 * 789?")
print(respuesta)
print("\n--- TOOL CALLS DETECTADAS ---")
print(respuesta.tool_calls)

### Celda 22: El ciclo completo, a mano

Este es el mecanismo interno que automatiza cualquier `AgentExecutor`:

1. El modelo decide qué tool usar (`tool_calls`).
2. Se ejecuta esa tool de verdad en Python, usando un diccionario
   `{"nombre": función}` como tabla de búsqueda para traducir el *nombre*
   (string) que devolvió el modelo a la función real.
3. El resultado se empaqueta en un `ToolMessage` (con `tool_call_id` para
   que el modelo sepa a qué petición corresponde cada resultado) y se
   agrega de vuelta a la conversación.
4. Se vuelve a invocar al modelo con la conversación actualizada, para que
   genere la respuesta final en lenguaje natural.

El modelo no puede ejecutar código por sí solo — solo genera texto
(incluyendo texto que dice "quiero usar esta tool"). Este ciclo es el
puente entre esa intención y la ejecución real.

In [ ]:
# ==============================================================================
# CELDA 22: Ciclo manual de tool calling (para entender el mecanismo)
# ==============================================================================
from langchain_core.messages import ToolMessage

mensajes = [HumanMessage("¿Cuánto es 345 * 789?")]

respuesta_ia = llm_con_tools.invoke(mensajes)
mensajes.append(respuesta_ia)

for tool_call in respuesta_ia.tool_calls:
    herramienta_seleccionada = {"calculadora": calculadora, "clima": clima}[tool_call["name"]]
    resultado_tool = herramienta_seleccionada.invoke(tool_call["args"])
    mensajes.append(ToolMessage(content=str(resultado_tool), tool_call_id=tool_call["id"]))

respuesta_final = llm_con_tools.invoke(mensajes)
print(respuesta_final.content)

### Nota sobre `create_tool_calling_agent` (camino explorado y descartado)

El lab original usa `create_react_agent` + `AgentExecutor` de
`langchain.agents`, que depende de que el modelo siga estrictamente el
formato de texto `Thought: / Action: / Action Input:` — frágil con modelos
pequeños como `llama3.2`.

Se intentó la alternativa moderna `create_tool_calling_agent` (basada en
tool calling nativo, igual que `bind_tools`), pero import falló con
`ImportError: cannot import name 'create_tool_calling_agent' from
'langchain.agents'` en la versión de `langchain` instalada — esa función
requiere una versión más reciente del paquete (`pip install --upgrade
langchain` la resolvería).

**Decisión tomada:** en vez de depender de una función que cambia de
ubicación/disponibilidad entre versiones de LangChain, se construyó un
"agente casero" (siguiente celda) que es, literalmente, el mismo ciclo de
la Celda 22 envuelto en un `for` con condición de salida. Es la misma
lógica que usa `AgentExecutor` internamente, sin la dependencia frágil de
versión. Si prefieres usar la función oficial, basta con
`pip install --upgrade langchain` y reiniciar el kernel antes de la Celda
20.

### Celda 23: Agente casero (`while`/`for` con condición de salida)

Automatiza el ciclo de la Celda 22: repite *"el modelo decide → se
ejecutan las tools pedidas → se le devuelve el resultado"* hasta que el
modelo ya no pida ninguna tool más (`tool_calls` vacío), momento en el que
`respuesta_ia.content` es la respuesta final.

> ⚠️ **No determinismo observado durante las pruebas:** con la pregunta
> multi-tool *"¿Cuánto es 100+50 y qué clima hace en Quito?"*, este mismo
> código, en distintas corridas, a veces detectó **ambas** tools
> correctamente en el primer paso, y otras veces detectó solo la
> calculadora y **alucinó** una respuesta de clima basada en su
> conocimiento de entrenamiento (mencionando incluso una fecha de corte),
> en vez de reconocer que necesitaba invocar la tool `clima`. Con
> `temperature=0.2` (no 0), hay algo de variabilidad entre corridas.
>
> **Recomendación para producción:** no valides un agente con una sola
> corrida. Corre la misma pregunta varias veces y mide la tasa de éxito
> real; considera agregar validación posterior (¿la respuesta final
> realmente usó todas las tools esperadas?) y, si la tasa de fallo es alta,
> evalúa modelos más grandes o más orientados a tool-calling antes de
> confiar el flujo a un modelo de 3B parámetros.

In [ ]:
# ==============================================================================
# CELDA 23: Agente casero (equivalente a AgentExecutor, sin dependencias frágiles)
# ==============================================================================
def mi_agente(pregunta, max_pasos=5, verbose=True):
    mensajes = [HumanMessage(pregunta)]
    mapa_tools = {"calculadora": calculadora, "clima": clima}

    for paso in range(max_pasos):
        respuesta_ia = llm_con_tools.invoke(mensajes)
        if verbose:
            print(f"--- PASO {paso} ---")
            print(f"Tool calls detectadas: {respuesta_ia.tool_calls}")
        mensajes.append(respuesta_ia)

        if not respuesta_ia.tool_calls:
            return respuesta_ia.content

        for tool_call in respuesta_ia.tool_calls:
            herramienta = mapa_tools[tool_call["name"]]
            resultado_tool = herramienta.invoke(tool_call["args"])
            mensajes.append(ToolMessage(content=str(resultado_tool), tool_call_id=tool_call["id"]))

    return "No se pudo resolver en el número máximo de pasos."


# Prueba única
print(mi_agente("¿Cuánto es 100+50 y qué clima hace en Quito?"))

### Celda 24 (opcional): Medir consistencia con corridas repetidas

Corre la misma pregunta varias veces para cuantificar, en vez de suponer,
qué tan seguido el agente detecta correctamente ambas herramientas. Es el
tipo de prueba que separa "probé el código y funcionó una vez" de
"entiendo cómo se comporta este sistema en la práctica".

In [ ]:
# ==============================================================================
# CELDA 24 (opcional): Prueba de consistencia del agente (varias corridas)
# ==============================================================================
N_CORRIDAS = 5
exitos_ambas_tools = 0

for i in range(N_CORRIDAS):
    print(f"\n===== CORRIDA {i+1}/{N_CORRIDAS} =====")
    respuesta = mi_agente(
        "¿Cuánto es 100+50 y qué clima hace en Quito?",
        verbose=False,
    )
    print(respuesta)

print(f"\n(Para medir la tasa de éxito real, revisa manualmente cuántas "
      f"respuestas mencionan tanto el resultado matemático como el clima "
      f"de Quito, en vez de asumir que siempre funciona igual.)")

## 12. Conclusiones y recomendaciones generales

### Lo que se logró
- Se replicó **todo** el recorrido conceptual del lab de IBM (modelo,
  chat/roles, prompt templates, salida estructurada, documents, loaders,
  splitters, embeddings, vector store, retriever, RAG, memoria, chains
  secuenciales, tools y agentes) **sin depender de ninguna API externa de
  pago**, usando exclusivamente Ollama en local.
- Se reemplazaron los patrones que el propio lab de IBM señala como en
  camino a deprecación (`LLMChain`, `SequentialChain`, `ConversationChain`,
  `RetrievalQA`, `create_react_agent`) por sus equivalentes modernos en
  LCEL (`prompt | llm | parser`, `RunnablePassthrough.assign`,
  `RunnableWithMessageHistory`, tool calling nativo con `bind_tools`).

### Diferencias reales frente a un modelo grande de pago (documentadas con evidencia, no solo supuestas)
1. **RAG por similarity search falla en preguntas de resumen global** y
   funciona bien en preguntas puntuales — esto es cierto para cualquier
   proveedor, pero se vuelve más evidente con retrievers simples y chunks
   pequeños como los usados aquí.
2. **`chunk_size` importa mucho:** chunks muy pequeños (200 caracteres)
   producen fragmentos rotos a mitad de oración que degradan la calidad de
   las respuestas de RAG.
3. **Encadenar la salida cruda de un LLM es frágil:** el relleno
   conversacional de un paso puede contaminar el prompt del siguiente paso
   si no se piden instrucciones de formato explícitas.
4. **El tool-calling de un modelo pequeño (3B) no es 100% consistente**
   corrida a corrida, incluso con `temperature` baja — a veces detecta
   correctamente todas las herramientas necesarias, a veces "alucina" en
   vez de invocar la herramienta correcta.

### Recomendaciones si esto se lleva a producción
- No confiar en una sola corrida de prueba para validar un agente o una
  chain — correr pruebas repetidas y medir tasas de éxito.
- Agregar pasos de validación/limpieza entre etapas de chains largas.
- Si la tasa de fallo de tool-calling es alta con un modelo pequeño, probar
  modelos locales más grandes (`llama3.1:70b` si el hardware lo permite) o
  modelos más orientados a tool-calling, antes de asumir que el diseño del
  pipeline está mal.
- Para RAG en documentos cortos, evaluar si realmente hace falta
  retrieval, o si simplemente pasar el documento completo da mejores
  resultados con menos complejidad.

---

*Notebook adaptado del lab de IBM/Skills Network "Build Smarter AI Apps:
Empower LLMs with LangChain" para correr 100% en local con Ollama.*
